In [1]:
!pip install ultralytics faster-coco-eval --no-dependencies --quiet

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 19.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 588.1/588.1 kB 33.0 MB/s eta 0:00:00


In [2]:
!mkdir -p /kaggle/working/data
!cp -r /kaggle/input/datasets/nttgaming/ripvis-cs431/datasets/train /kaggle/working/data/train
!cp -r /kaggle/input/datasets/nttgaming/ripvis-cs431/datasets/test /kaggle/working/data/test

In [3]:
import os
os.environ['PYTORCH_KERNEL_CACHE_PATH'] = './torch_kernel_cache'
os.makedirs('./torch_kernel_cache', exist_ok=True)

In [4]:
import pandas as pd
import numpy as np
import json
import yaml
from ultralytics import YOLO, settings
from pycocotools.coco import COCO
from pycocotools import mask as maskUtils
import torch
import random

FOLD_CSV = '/kaggle/input/datasets/nttgaming/ripvis-cs431/fold_management.csv'
DATASET_PATH = '/kaggle/working/data/train'
IOU_THRESHOLDS = np.arange(0.40, 0.96, 0.05)
SEED = 42

Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.


In [5]:
def set_seed(seed=42):
    random.seed(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

set_seed(SEED)

In [6]:
def compute_composite_score(gt_json_path, pred_json_path):
    if not os.path.exists(pred_json_path): return {"Score": 0.0}
    
    gt_coco = COCO(gt_json_path)
    with open(pred_json_path) as f: preds = json.load(f)
    pred_coco = gt_coco.loadRes(preds)
    
    # Evaluate SEG
    counts = [[0, 0, 0] for _ in IOU_THRESHOLDS]
    for img_id in gt_coco.getImgIds():
        gt_anns = [a for a in gt_coco.loadAnns(gt_coco.getAnnIds(imgIds=img_id)) if "segmentation" in a]
        pr_anns = sorted([a for a in pred_coco.loadAnns(pred_coco.getAnnIds(imgIds=img_id)) if "segmentation" in a], 
                         key=lambda x: x.get("score", 0), reverse=True)
        
        if not gt_anns and not pr_anns: continue
        
        # IoU Matching logic (RLE)
        gt_rle = [gt_coco.annToRLE(a) for a in gt_anns]
        pr_rle = [gt_coco.annToRLE(a) for a in pr_anns]
        
        if not pr_anns:
            for idx in range(len(IOU_THRESHOLDS)): counts[idx][2] += len(gt_anns)
            continue
        if not gt_anns:
            for idx in range(len(IOU_THRESHOLDS)): counts[idx][1] += len(pr_anns)
            continue

        ious = maskUtils.iou(pr_rle, gt_rle, np.zeros(len(gt_rle)))
        for idx, thr in enumerate(IOU_THRESHOLDS):
            matched_gt = np.zeros(len(gt_anns), dtype=bool)
            tp = fp = 0
            for i in range(len(pr_anns)):
                eligible = (ious[i] >= thr) & (~matched_gt)
                if eligible.any():
                    tp += 1
                    matched_gt[np.argmax(np.where(eligible, ious[i], -1.0))] = True
                else: fp += 1
            counts[idx][0] += tp; counts[idx][1] += fp; counts[idx][2] += int((~matched_gt).sum())

    # Calculate Metrics
    f1_list, f2_list = [], []
    for tp, fp, fn in counts:
        p = tp / (tp + fp) if (tp + fp) > 0 else 0
        r = tp / (tp + fn) if (tp + fn) > 0 else 0
        f1_list.append(2*p*r/(p+r) if (p+r)>0 else 0)
        f2_list.append(5*p*r/(4*p+r) if (4*p+r)>0 else 0)

    # Lấy F1 và F2 tại ngưỡng IoU = 0.50 (index 2 trong mảng 0.40, 0.45, 0.50...)
    idx_50 = 2
    f1_50 = f1_list[idx_50]
    f2_50 = f2_list[idx_50]
    
    # Tính mean cho toàn bộ ngưỡng (0.40 đến 0.95)
    f1_4095 = float(np.mean(f1_list))
    f2_4095 = float(np.mean(f2_list))

    # Composite: 0.25*(F1@50 + F2@50 + mean_F1 + mean_F2)
    score = 0.25 * f1_50 + 0.25 * f2_50 + 0.25 * f1_4095 + 0.25 * f2_4095

    return {
        "F1[50]": float(f1_50),
        "F2[50]": float(f2_50),
        "F1[40:95]": float(f1_4095),
        "F2[40:95]": float(f2_4095),
        "Score": float(score)
    }

In [7]:
import wandb
from kaggle_secrets import UserSecretsClient

In [8]:
wandb.login(key=UserSecretsClient().get_secret("wandb"))
settings.update({"wandb": True})

wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: [wandb.login()] Using explicit session credentials for https://api.wandb.ai.
wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: minhhoanghsftg to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


In [9]:
FOLD_NUM = 0

In [10]:
df = pd.read_csv(FOLD_CSV)
train_df = df[df['fold'] != FOLD_NUM]
val_df = df[df['fold'] == FOLD_NUM]

# Tạo file list cho YOLO
train_list = f'train_f{FOLD_NUM}.txt'
val_list = f'val_f{FOLD_NUM}.txt'
test_list = f'test_f{FOLD_NUM}.txt'

with open(train_list, 'w') as f:
    f.write('\n'.join([os.path.join(DATASET_PATH, 'images', name) for name in train_df['file_name']]))
with open(val_list, 'w') as f:
    f.write('\n'.join([os.path.join(DATASET_PATH, 'images', name) for name in val_df['file_name']]))

# Tạo GT JSON cho Fold này để đánh giá
with open(os.path.join(DATASET_PATH, 'train_with_additional_data.json'), 'r') as f:
    full_coco = json.load(f)

val_fns = set(val_df['file_name'])
fold_gt = {
    "images": [i for i in full_coco['images'] if i['file_name'] in val_fns],
    "annotations": [],
    "categories": full_coco['categories']
}
v_ids = set(i['id'] for i in fold_gt['images'])
fold_gt['annotations'] = [a for a in full_coco['annotations'] if a['image_id'] in v_ids]

gt_json_fold = f'gt_fold_{FOLD_NUM}.json'
with open(gt_json_fold, 'w') as f:
    json.dump(fold_gt, f)

# Cấu hình YAML
yaml_p = f'rip_f{FOLD_NUM}.yaml'
with open(yaml_p, 'w') as f:
    yaml.dump({'path': '/kaggle/working', 'train': train_list, 'val': val_list, 'test': 'data/test/images', 'nc': 1, 'names': ['rip']}, f)

In [11]:
# ============================================================
# PATCH: ultralytics tasks.py — inject CBAM support
# Works on Kaggle user-installed ultralytics (clean env)
# ============================================================
import re
from pathlib import Path

tasks_path = Path("/usr/local/lib/python3.12/dist-packages/ultralytics/nn/tasks.py")
src = tasks_path.read_text()

# ── PATCH 1: Force CBAM into the module import block ────────
# Find any existing import from ultralytics.nn.modules and inject CBAM
# Strategy: locate the block-import line containing "CBLinear" OR any nearby anchor
if "CBAM," not in src:
    # Try anchoring on CBLinear (most likely neighbor)
    if "CBLinear," in src:
        src = src.replace("CBLinear,", "CBAM,\n    CBLinear,", 1)
    else:
        # Fallback: anchor on the closing of the nn.modules import block
        # Find last item before the closing paren of the from ... import (... ) block
        src = re.sub(
            r'(from ultralytics\.nn\.modules[^\)]*?)(C2f,)',
            r'\1CBAM,\n    C2f,',
            src,
            count=1,
            flags=re.DOTALL,
        )

print("PATCH 1 (import) applied:", "CBAM," in src)

# ── PATCH 2: Handle CBAM channel logic in parse_model ───────
# Target the generic fallback `else: c2 = ch[f]`
# We inject a CBAM-specific branch before it
cbam_branch = (
    "        elif m is CBAM:\n"
    "            c2 = ch[f]\n"
    "            args = [c2]\n"
)

if "elif m is CBAM:" not in src:
    # Match the exact else-block that sets c2 = ch[f] as the generic fallback
    pattern = r'(        else:\n            c2 = ch\[f\])'
    replacement = cbam_branch + r'\1'
    new_src, n = re.subn(pattern, replacement, src, count=1)
    if n:
        src = new_src
        print("PATCH 2 (parse_model branch) applied: True")
    else:
        print("PATCH 2 WARNING: pattern not found, dumping nearby context for debug")
        idx = src.find("c2 = ch[f]")
        print(repr(src[max(0, idx-200):idx+100]))
else:
    print("PATCH 2 already present, skipping")

tasks_path.write_text(src)
print("tasks.py written ✓")


# ── PATCH 3: Guarantee CBAM lands in globals() at runtime ───
# Belt-and-suspenders: monkey-patch the live module so the
# already-imported ultralytics session also sees CBAM
import ultralytics.nn.tasks as _tasks
from ultralytics.nn.modules import CBAM as _CBAM   # will raise if CBAM missing from modules
import builtins

# Inject into the tasks module's own global namespace
_tasks.__dict__["CBAM"] = _CBAM
print("PATCH 3 (runtime globals injection) applied ✓")


# ── VERIFY ──────────────────────────────────────────────────
assert "CBAM," in tasks_path.read_text(),          "❌ CBAM import line missing"
assert "elif m is CBAM:" in tasks_path.read_text(), "❌ parse_model branch missing"
print("\n✅ All patches verified. Reload ultralytics before building the model.")


# ── RELOAD ultralytics so patched file takes effect ─────────
import importlib, ultralytics.nn.tasks, ultralytics.nn.modules
importlib.reload(ultralytics.nn.modules)
importlib.reload(ultralytics.nn.tasks)

# Re-inject after reload (reload wipes the dict)
import ultralytics.nn.tasks as _tasks
from ultralytics.nn.modules import CBAM as _CBAM
_tasks.__dict__["CBAM"] = _CBAM
print("Module reloaded and CBAM re-injected ✓")

PATCH 1 (import) applied: True
PATCH 2 (parse_model branch) applied: True
tasks.py written ✓
PATCH 3 (runtime globals injection) applied ✓

✅ All patches verified. Reload ultralytics before building the model.
Module reloaded and CBAM re-injected ✓


In [12]:
model_yaml_content = """ 
# Parameters
nc: 1 # number of classes
end2end: True # whether to use end-to-end mode
reg_max: 1 # DFL bins
scales: # model compound scaling constants, i.e. 'model=yolo26n-seg.yaml' will call yolo26-seg.yaml with scale 'n'
  # [depth, width, max_channels]
  n: [0.50, 0.25, 1024]
  s: [0.50, 0.50, 1024]
  m: [0.50, 1.00, 512]
  l: [1.00, 1.00, 512]
  x: [1.00, 1.50, 512]

# YOLO26n backbone
backbone:
  # [from, repeats, module, args]
  - [-1, 1, Conv, [64, 3, 2]]                   # 0-P1/2
  - [-1, 1, Conv, [128, 3, 2]]                  # 1-P2/4
  - [-1, 2, C3k2, [256, False, 0.25]]
  - [-1, 1, Conv, [256, 3, 2]]                  # 3-P3/8
  - [-1, 2, C3k2, [512, False, 0.25]]
  - [-1, 1, Conv, [512, 3, 2]]                  # 5-P4/16
  - [-1, 2, C3k2, [512, True]]
  - [-1, 1, Conv, [1024, 3, 2]]                 # 7-P5/32
  - [-1, 2, C3k2, [1024, True]]
  - [-1, 1, SPPF, [1024, 5, 3, True]]           # 9
  - [-1, 2, C2PSA, [1024]]                      # 10

# YOLO26n head
head:  
  - [-1, 1, nn.Upsample, [None, 2, "nearest"]]  # 11
  - [[-1, 6], 1, Concat, [1]]                   # 12 - cat backbone P4
  - [-1, 2, C3k2, [512, True]]                  # 13

  - [-1, 1, nn.Upsample, [None, 2, "nearest"]]  # 14
  - [[-1, 4], 1, Concat, [1]]                   # 15
  - [-1, 2, C3k2, [256, True]]                  # 16 (P3/8-small)
  - [-1, 1, CBAM, []]                           # 17

  - [-1, 1, Conv, [256, 3, 2]]                  # 18
  - [[-1, 13], 1, Concat, [1]]                  # 19
  - [-1, 2, C3k2, [512, True]]                  # 20 (P4/16-medium)
  - [-1, 1, CBAM, []]                           # 21

  - [-1, 1, Conv, [512, 3, 2]]                  # 22
  - [[-1, 10], 1, Concat, [1]]                  # 23
  - [-1, 1, C3k2, [1024, True, 0.5, True]]      # 24 (P5/32-large)

  - [[17, 21, 24], 1, Segment26, [nc, 32, 256]] # 26
"""

with open("/kaggle/working/yolo26-seg-cbam-p3p4.yaml", "w") as f:
    f.write(model_yaml_content)

In [13]:
model = YOLO("/kaggle/working/yolo26n-seg-cbam-p3p4.yaml").load("yolo26n-seg.pt")

Transferred 348/850 items from pretrained weights


In [14]:
train_args = {
    "data": yaml_p,
    "epochs": 150,
    "patience": 20,
    "save": True,
    "device": [0, 1],
    "cos_lr": True,
    "batch": 128,
    "workers": 2,
    "val": True,
    "imgsz": 640,
    "warmup_epochs": 5,
    "seed": SEED,
    # "compile": True,
    "max_det": 50,
    "project": "UIT-RipVis",
    "name": f"YOLO26n-seg-CBAM_P3P4-{FOLD_NUM}",
}

In [15]:
model.train(**train_args)

Ultralytics 8.4.41 🚀 Python-3.12.12 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
                                                       CUDA:1 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=128, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=True, cutmix=0.0, data=rip_f0.yaml, degrees=0.0, deterministic=True, device=0,1, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=150, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=50, mixup=0.0, mode=train, model=/kaggle/working/yolo26n-seg-cbam-p3p4.yaml, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=YOLO26

wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from /root/.netrc.
wandb: Currently logged in as: minhhoanghsftg to https://api.wandb.ai. Use `wandb login --relogin` to force relogin
wandb: setting up run YOLO26n-seg-CBAM_P3P4-0_20260427_082550
wandb: Tracking run with wandb version 0.25.0
wandb: Run data is saved locally in /kaggle/working/runs/segment/UIT-RipVis/YOLO26n-seg-CBAM_P3P4-0/wandb/run-20260427_082551-YOLO26n-seg-CBAM_P3P4-0_20260427_082550
wandb: Run `wandb offline` to turn off syncing.
wandb: Syncing run YOLO26n-seg-CBAM_P3P4-0
wandb: ⭐️ View project at https://wandb.ai/minhhoanghsftg/UIT-RipVis
wandb: 🚀 View run at https://wandb.ai/minhhoanghsftg/UIT-RipVis/runs/YOLO26n-seg-CBAM_P3P4-0_20260427_082550


Transferred 348/850 items from pretrained weights
AMP: running Automatic Mixed Precision (AMP) checks...
AMP: checks passed ✅
train: Fast image access ✅ (ping: 0.0±0.0 ms, read: 2062.6±803.0 MB/s, size: 125.1 KB)
train: Scanning /kaggle/working/data/train/labels... 10880 images, 5670 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 16550/16550 1.1Kit/s 14.6s
train: New cache created: /kaggle/working/data/train/labels.cache
albumentations: Blur(p=0.01, blur_limit=(3, 7)), MedianBlur(p=0.01, blur_limit=(3, 7)), ToGray(p=0.01, method='weighted_average', num_output_channels=3), CLAHE(p=0.01, clip_limit=(1.0, 4.0), tile_grid_size=(8, 8))
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 876.2±561.3 MB/s, size: 171.6 KB)
val: Scanning /kaggle/working/data/train/labels... 1209 images, 630 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 1839/1839 861.2it/s 2.1s
val: New cache created: /kaggle/working/data/train/labels.cache
optimizer: 'optimizer=auto' found, ignoring 'lr0=0.01' and 'momentum=0.937' and det

wandb: WARNING Tried to log to step 150 that is less than the current step 151. Steps must be monotonically increasing, so this data will be ignored. See https://wandb.me/define-metric to log data out of order.
wandb: uploading artifact run_YOLO26n-seg-CBAM_P3P4-0_20260427_082550_model; updating run metadata; uploading artifact run-YOLO26n-seg-CBAM_P3P4-0_20260427_082550-curvesPrecision-RecallB_table-zAfnsw; uploading artifact run-YOLO26n-seg-CBAM_P3P4-0_20260427_082550-curvesF1-ConfidenceB_table-WsQ7TA; uploading artifact run-YOLO26n-seg-CBAM_P3P4-0_20260427_082550-curvesPrecision-ConfidenceB_table-jAlSEQ (+ 5 more)
wandb: uploading artifact run_YOLO26n-seg-CBAM_P3P4-0_20260427_082550_model; uploading artifact run-YOLO26n-seg-CBAM_P3P4-0_20260427_082550-curvesPrecision-RecallB_table-zAfnsw; uploading artifact run-YOLO26n-seg-CBAM_P3P4-0_20260427_082550-curvesF1-ConfidenceB_table-WsQ7TA; uploading artifact run-YOLO26n-seg-CBAM_P3P4-0_20260427_082550-curvesPrecision-ConfidenceB_table-jA

In [16]:
settings.update({"wandb": False})

In [17]:
model = YOLO(f"/kaggle/working/runs/segment/UIT-RipVis/YOLO26n-seg-CBAM_P3P4-{FOLD_NUM}/weights/best.pt")

In [18]:
def evaluate(pred_json_path, gt_json_path):
    # Kiểm tra xem YOLO có sinh ra file không
    if not os.path.exists(pred_json_path):
        print(f"Không tìm thấy {pred_json_path}. Trả về Score 0.0")
        score_results = {"Score": 0.0, "F1[50]": 0.0, "F2[50]": 0.0, "F1[40:95]": 0.0, "F2[40:95]": 0.0}
    else:
        # 1. Tạo từ điển map Tên file -> ID số từ biến fold_gt đã tạo ở trên
        filename_to_id = {os.path.splitext(img['file_name'])[0]: img['id'] for img in fold_gt['images']}
        
        # 2. Mở file JSON dự đoán
        with open(pred_json_path, 'r') as f:
            preds = json.load(f)
            
        # 3. Đổi ID chữ thành ID số
        valid_preds = []
        for p in preds:
            stem = str(p['image_id']) # ví dụ: "RipVIS-141_00060"
            if stem in filename_to_id:
                p['image_id'] = filename_to_id[stem] # Gán lại ID số nguyên
                valid_preds.append(p)
                
        # 4. Ghi đè lại file
        with open(pred_json_path, 'w') as f:
            json.dump(valid_preds, f)
            
        # 5. Đưa file ĐÃ SỬA vào hàm tính điểm
        score_results = compute_composite_score(gt_json_path, pred_json_path)
        
    score_results['fold'] = FOLD_NUM
    
    json_result_path = f'test_result_fold_{FOLD_NUM}.json'
    with open(json_result_path, 'w') as f:
        json.dump(score_results, f, indent=4)

    return score_results

In [19]:
model.val(data=yaml_p, split='val', save_json=True, project='Eval', name=f'fold_{FOLD_NUM}', conf=0.25)

pred_json_path = f'/kaggle/working/runs/segment/Eval/fold_{FOLD_NUM}/predictions.json'

try:
    score_results = evaluate(
        pred_json_path, gt_json_fold
    )
    print(score_results)
except Exception:
    print("Oh on, something went wrong")

Ultralytics 8.4.41 🚀 Python-3.12.12 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
YOLO26n-seg-cbam-p3p4 summary (fused): 149 layers, 2,709,947 parameters, 0 gradients
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 2732.9±469.8 MB/s, size: 144.4 KB)
val: Scanning /kaggle/working/data/train/labels.cache... 1209 images, 630 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 1839/1839 308.5Mit/s 0.0s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95)     Mask(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 115/115 4.4it/s 26.0s
                   all       1839       1829      0.951      0.952      0.968      0.736      0.954      0.955      0.968      0.668
Speed: 0.9ms preprocess, 6.2ms inference, 0.0ms loss, 1.3ms postprocess per image
Saving /kaggle/working/runs/segment/Eval/fold_0/predictions.json...
Results saved to /kaggle/working/runs/segment/Eval/fold_0
loading annotations into memory...
Done (t=0.05s)
creating index...
index created!
Loading

In [20]:
def evaluate_test_only(pred_json_path, gt_json_path):
    if not os.path.exists(pred_json_path):
        return {"Score": 0.0, "F1[50]": 0.0, "F2[50]": 0.0, "F1[40:95]": 0.0, "F2[40:95]": 0.0}

    with open(pred_json_path, 'r') as f:
        preds = json.load(f)
        
    if not preds:
        return {"Score": 0.0, "F1[50]": 0.0, "F2[50]": 0.0, "F1[40:95]": 0.0, "F2[40:95]": 0.0}

    with open(gt_json_path, 'r') as f:
        test_coco = json.load(f)

    test_id_map = {os.path.splitext(img['file_name'])[0]: img['id'] for img in test_coco['images']}

    valid_preds = []
    for p in preds:
        pred_stem = str(p['image_id'])
        if pred_stem in test_id_map:
            p['image_id'] = test_id_map[pred_stem]
            valid_preds.append(p)

    if not valid_preds:
        return {"Score": 0.0, "F1[50]": 0.0, "F2[50]": 0.0, "F1[40:95]": 0.0, "F2[40:95]": 0.0}

    with open(pred_json_path, 'w') as f:
        json.dump(valid_preds, f)

    return compute_composite_score(gt_json_path, pred_json_path)

In [21]:
model.val(data=yaml_p, split='test', save_json=True, project='Test', name=f'fold_{FOLD_NUM}', conf=0.25)

pred_json_path = f'/kaggle/working/runs/segment/Test/fold_{FOLD_NUM}/predictions.json'

try:
    score_results = evaluate_test_only(
        pred_json_path, "/kaggle/input/datasets/lakechamplain/ripvis-cs431-test-gt/test_gt.json"
    )
    print(score_results)
except Exception:
    print("Oh on, something went wrong")

Ultralytics 8.4.41 🚀 Python-3.12.12 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 2724.8±605.8 MB/s, size: 160.2 KB)
val: Scanning /kaggle/working/data/test/labels... 2861 images, 1488 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 4349/4349 1.3Kit/s 3.3s
val: New cache created: /kaggle/working/data/test/labels.cache
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95)     Mask(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 272/272 5.0it/s 54.9s
                   all       4349       4556      0.785      0.578      0.535      0.252       0.79      0.582      0.546      0.246
Speed: 0.9ms preprocess, 6.3ms inference, 0.0ms loss, 1.0ms postprocess per image
Saving /kaggle/working/runs/segment/Test/fold_0/predictions.json...
Results saved to /kaggle/working/runs/segment/Test/fold_0
loading annotations into memory...
Done (t=0.14s)
creating index...
index created!
Loading and preparing results...
DONE

In [22]:
!rm -r /kaggle/working/data